# Step 1: Download RSNA Pneumonia data and explore DICOM
Before running: Runtime > Change runtime type > T4 GPU. Also accept the competition rules at kaggle.com/c/rsna-pneumonia-detection-challenge (Rules tab), otherwise download fails with 403.

In [ ]:
!pip -q install pydicom kaggle

In [ ]:
# Upload kaggle.json (the API token) so the kaggle CLI can authenticate.
from google.colab import files
import os, shutil
files.upload()  # pick kaggle.json in the dialog
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)  # kaggle refuses world-readable keys

In [ ]:
# Download to local disk /content (fast), NOT Google Drive (slow).
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna
!cd /content/rsna && unzip -q rsna-pneumonia-detection-challenge.zip && ls

In [ ]:
import pandas as pd
labels = pd.read_csv('/content/rsna/stage_2_train_labels.csv')
print(labels.head())
# One row per bounding box; a patient can have several boxes or none.
per_patient = labels.groupby('patientId')['Target'].max()
print(per_patient.value_counts())  # class balance: 0 = normal/not pneumonia, 1 = pneumonia

In [ ]:
import pydicom, matplotlib.pyplot as plt
pid = per_patient[per_patient==1].index[0]
ds = pydicom.dcmread(f'/content/rsna/stage_2_train_images/{pid}.dcm')
print(ds)  # shows every DICOM tag: the header metadata

In [ ]:
import numpy as np, matplotlib.patches as patches
img = ds.pixel_array
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(img, cmap='gray')
for _, r in labels[(labels.patientId==pid)&(labels.Target==1)].iterrows():
    ax.add_patch(patches.Rectangle((r.x, r.y), r.width, r.height, fill=False, edgecolor='red', lw=2))
ax.set_title(f'{pid} pneumonia boxes in red'); plt.show()
print(img.shape, img.dtype, img.min(), img.max())